<a href="https://colab.research.google.com/github/ARNAVKS/Named-Entity-Recognition/blob/main/Tag_improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NER Tagging — Improved Notebook
**Changes from original:**
- Proper 80/10/10 train/val/test split
- Tokenizer fit only on train (no leakage)
- MaskedAccuracy metric (excludes padding tokens)
- EarlyStopping + ModelCheckpoint
- seqeval per-entity F1 on held-out test set
- Clean inference function


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d naseralqaydeh/named-entity-recognition-ner-corpus
from zipfile import ZipFile
with ZipFile('named-entity-recognition-ner-corpus.zip','r') as f:
    f.extractall()

Dataset URL: https://www.kaggle.com/datasets/naseralqaydeh/named-entity-recognition-ner-corpus
License(s): DbCL-1.0
100% 4.14M/4.14M [00:00<00:00, 39.9MB/s]



In [ ]:
!pip install seqeval -q

import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from seqeval.metrics import classification_report
print('TF:', tf.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
TF: 2.20.0


## Load & Parse Data

In [ ]:
import ast
table = pd.read_csv('ner.csv')
table['Tag'] = table['Tag'].apply(ast.literal_eval)

# Tag column is space-separated string e.g. 'O O B-geo O'
# No ast.literal_eval needed unlike POS column
table['words']    = table['Sentence'].apply(lambda x: x.split())
table = table[[len(table['words'][i]) == len(table['Tag'][i]) for i in range(len(table))]].reset_index(drop=True)
print(f'After dropping mismatches: {len(table)}')  # should be 47955
# Sanity check
mismatches = [i for i in range(len(table)) if len(table['words'][i]) != len(table['Tag'][i])]
print(f'Mismatches: {len(mismatches)}')  # should be 0
print(f'Total sentences: {len(table)}')
table[['Sentence','Tag']].head(3)

After dropping mismatches: 47955
Mismatches: 0
Total sentences: 47955


,Sentence,Tag
0,Thousands of demonstrators have marched throug...,"[O, O, O, O, O, O, B-geo, O, O, O, O, O, B-geo..."
1,Families of soldiers killed in the conflict jo...,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
2,They marched from the Houses of Parliament to ...,"[O, O, O, O, O, O, O, O, O, O, O, B-geo, I-geo..."


In [ ]:
table['Tag'][0]

['O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-geo',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-geo',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-gpe',
 'O',
 'O',
 'O',
 'O',
 'O']

## Train / Val / Test Split

In [ ]:
train_df, test_df = train_test_split(table, test_size=0.10, random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.10, random_state=42)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

Train: 38843 | Val: 4316 | Test: 4796


## Tokenization (fit on train only)

In [ ]:
max_len = max(len(s) for s in table['words'])
print('max_len:', max_len)

import pickle
with open('sent_token (1).pkl','rb') as t:
  sent_tok=pickle.load(t)
ner_tok  = Tokenizer(filters='', lower=False)

# Fit only on train — avoids test vocabulary leaking into training
sent_tok.fit_on_texts(train_df['Sentence'])
ner_tok.fit_on_texts(train_df['Tag'])

print(f'Vocab: {len(sent_tok.word_index)} | NER tags: {len(ner_tok.word_index)}')
print('Tag vocab:', ner_tok.word_index)

max_len: 104
Vocab: 31362 | NER tags: 17
Tag vocab: {'O': 1, 'B-geo': 2, 'B-tim': 3, 'B-org': 4, 'I-per': 5, 'B-per': 6, 'I-org': 7, 'B-gpe': 8, 'I-geo': 9, 'I-tim': 10, 'B-art': 11, 'B-eve': 12, 'I-art': 13, 'I-eve': 14, 'I-gpe': 15, 'B-nat': 16, 'I-nat': 17}


In [ ]:
def encode(df):
    X = pad_sequences(sent_tok.texts_to_sequences(df['Sentence']),
                      maxlen=max_len, padding='post')
    y = pad_sequences(ner_tok.texts_to_sequences(df['Tag']),
                      maxlen=max_len, padding='post')
    return X, y

X_train, y_train = encode(train_df)
X_val,   y_val   = encode(val_df)
X_test,  y_test  = encode(test_df)
print(X_train.shape, y_train.shape)

(38843, 104) (38843, 104)


## Model with MaskedAccuracy
> Standard accuracy counts padding (label=0) as correct and inflates the metric. MaskedAccuracy only counts real tokens.

In [ ]:
class MaskedAccuracy(tf.keras.metrics.Metric):
    def __init__(self, **kwargs):
        super().__init__(name='masked_accuracy', **kwargs)
        self.correct = self.add_weight(name='correct', shape=(), initializer='zeros')
        self.total   = self.add_weight(name='total',   shape=(), initializer='zeros')
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred_ids = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        y_true     = tf.cast(y_true, tf.int32)
        mask       = tf.not_equal(y_true, 0)
        correct    = tf.logical_and(tf.equal(y_true, y_pred_ids), mask)
        self.correct.assign_add(tf.cast(tf.reduce_sum(tf.cast(correct, tf.int32)), tf.float32))
        self.total.assign_add(tf.cast(tf.reduce_sum(tf.cast(mask, tf.int32)), tf.float32))
    def result(self):
        return self.correct / (self.total + 1e-8)
    def reset_state(self):
        self.correct.assign(0.0)
        self.total.assign(0.0)

num_ner_tags = len(ner_tok.word_index) + 1
vocab_size   = len(sent_tok.word_index) + 1

tag_model = models.Sequential([
    layers.Embedding(vocab_size, 100, mask_zero=True),
    layers.Bidirectional(layers.LSTM(64, dropout=0.2, return_sequences=True)),
    layers.TimeDistributed(layers.Dense(num_ner_tags, activation='softmax'))
])
tag_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=[MaskedAccuracy()]
)
tag_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Train

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_masked_accuracy', patience=2,
                  restore_best_weights=True, mode='max'),
    ModelCheckpoint('tag_model.keras', monitor='val_masked_accuracy',
                    save_best_only=True, verbose=1, mode='max')
]

history = tag_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32,
    callbacks=callbacks
)

Epoch 1/5
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step - loss: 0.6011 - masked_accuracy: 0.8776
Epoch 1: val_masked_accuracy improved from None to 0.95461, saving model to tag_model.keras

Epoch 1: finished saving model to tag_model.keras
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 404s 322ms/step - loss: 0.3269 - masked_accuracy: 0.9186 - val_loss: 0.1554 - val_masked_accuracy: 0.9546
Epoch 2/5
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step - loss: 0.1269 - masked_accuracy: 0.9623
Epoch 2: val_masked_accuracy improved from 0.95461 to 0.96109, saving model to tag_model.keras

Epoch 2: finished saving model to tag_model.keras
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 382s 315ms/step - loss: 0.1244 - masked_accuracy: 0.9627 - val_loss: 0.1315 - val_masked_accuracy: 0.9611
Epoch 3/5
1214/1214 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step - loss: 0.0949 - masked_accuracy: 0.9706
Epoch 3: val_masked_accuracy improved from 0.96109 to 0.96133, saving model to tag_model.keras

Epoch 3: finished saving model to tag_model.keras
121

## Evaluate on Test Set with seqeval

In [ ]:
def get_seqeval_preds(model, X, y_true_ids):
    idx2ner = ner_tok.index_word   # {1: 'O', 2: 'B-geo', ...}
    preds   = np.argmax(model.predict(X, batch_size=64), axis=-1)
    true_seqs, pred_seqs = [], []
    for true_row, pred_row in zip(y_true_ids, preds):
        tl, pl = [], []
        for t, p in zip(true_row, pred_row):
            if t != 0:                             # skip padding
                tl.append(idx2ner.get(t, 'O'))
                pl.append(idx2ner.get(p, 'O'))
        true_seqs.append(tl)
        pred_seqs.append(pl)
    return true_seqs, pred_seqs

true_seqs, pred_seqs = get_seqeval_preds(tag_model, X_test, y_test)
print('=== NER Test Results ===')
print(classification_report(true_seqs, pred_seqs))

75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 114ms/step
=== NER Test Results ===


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

         art       0.00      0.00      0.00        42
         eve       0.50      0.26      0.34        27
         geo       0.82      0.88      0.85      3808
         gpe       0.96      0.94      0.95      1547
         nat       0.70      0.27      0.39        26
         org       0.59      0.58      0.59      1937
         per       0.72      0.70      0.71      1653
         tim       0.86      0.85      0.85      2003

   micro avg       0.79      0.80      0.79     11043
   macro avg       0.64      0.56      0.58     11043
weighted avg       0.79      0.80      0.79     11043



## Inference

In [ ]:
def tag(sentence):
    sent     = sent_tok.texts_to_sequences([sentence])
    sent_pad = pad_sequences(sent, maxlen=max_len, padding='post')
    pred     = tag_model.predict(sent_pad)
    pred_ids = np.argmax(pred, -1)[0]
    words    = sentence.split()
    tags     = [ner_tok.index_word.get(pred_ids[i], 'O') for i in range(len(words))]
    print(f'Sentence : {sentence}')
    print(f'NER      : {tags}')
tag(table['Sentence'][0])
tag('Barack Obama visited Paris in July to meet with the WHO representatives.')
tag('Google opened a new office in New York on Monday .')
tag('She read The New York Times article about NASA .')
tag('Despite being tired , she completed the assignment on time')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
Sentence : Thousands of demonstrators have marched through London to protest the war in Iraq and demand the withdrawal of British troops from that country .
NER      : ['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-gpe', 'O', 'O', 'O', 'O', 'O']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
Sentence : Barack Obama visited Paris in July to meet with the WHO representatives.
NER      : ['B-per', 'B-per', 'O', 'B-geo', 'O', 'B-tim', 'O', 'O', 'O', 'O', 'O', 'I-org']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
Sentence : Google opened a new office in New York on Monday .
NER      : ['B-org', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'I-geo', 'O', 'B-tim', 'O']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step
Sentence : She read The New York Times article about NASA .
NER      : ['O', 'O', 'O', 'B-org', 'I-org', 'I-org', 'O', 'O', 'B-org', 'O']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step
Sentence : Despite being tired , she complete

## Save

In [ ]:
pickle.dump(ner_tok,  open('ner_token.pkl',  'wb'))
tag_model.save('tag_model.keras')
print('Saved: tag_model.keras, sent_token.pkl, ner_token.pkl')

Saved: tag_model.keras, sent_token.pkl, ner_token.pkl
